# Preprocessing

Produces a clean, tokenized version of every speech that can be reused for LDA, TF-IDF, clustering, and association mining

## Setup

In [ ]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Project paths
PROJECT_DIR = Path.cwd()

# If running from the notebooks/ directory, move up one level
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "TXT"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Input directory:", DATA_DIR)
print("Output directory:", PROCESSED_DIR)

assert DATA_DIR.exists(), f"Could not find corpus at {DATA_DIR}"

In [ ]:
# Find all text files
text_files = sorted(DATA_DIR.rglob("*.txt"))

print(f"Found {len(text_files):,} text files")

# Extract country, session, and year from filenames
filename_pattern = re.compile(r"([A-Z]{2,3})_(\d+)_(\d{4})\.txt$")
folder_pattern = re.compile(r"Session\s+(\d+)\s*-\s*(\d{4})")

records = []

for path in text_files:
    filename_match = filename_pattern.search(path.name)
    folder_match = folder_pattern.search(path.parent.name)

    # Skip files that do not follow the expected naming convention
    if filename_match is None:
        continue

    country_code, session, year = filename_match.groups()

    records.append({
        "file_path": str(path),
        "filename": path.name,
        "country_code": country_code,
        "session": int(session),
        "year": int(year),
        "folder": path.parent.name,
        "folder_session": int(folder_match.group(1)) if folder_match else None,
        "folder_year": int(folder_match.group(2)) if folder_match else None
    })

metadata = pd.DataFrame(records)

print(f"Valid speech files: {len(metadata):,}")
print(f"Countries/entities: {metadata['country_code'].nunique():,}")
print(f"Year range: {metadata['year'].min()}–{metadata['year'].max()}")

display(metadata.head())

## Read speech text and handle encoding

In [ ]:
def read_speech(path):
    """
    Read a speech using UTF-8, falling back to CP1252
    if UTF-8 decoding fails.
    """
    path = Path(path)

    try:
        text = path.read_text(encoding="utf-8")
        encoding_used = "utf-8"
    except UnicodeDecodeError:
        text = path.read_text(encoding="cp1252")
        encoding_used = "cp1252"

    return text, encoding_used


texts = []
encodings = []

for path in metadata["file_path"]:
    text, encoding_used = read_speech(path)
    texts.append(text)
    encodings.append(encoding_used)

metadata["raw_text"] = texts
metadata["encoding"] = encodings

print("Encoding counts:")
display(metadata["encoding"].value_counts())

print("\nExample speech:")
print(metadata.loc[0, "raw_text"][:1500])

In [ ]:
# Inspect text quality after reading
def count_replacement_chars(text):
    return text.count("\ufffd")


metadata["replacement_char_count"] = metadata["raw_text"].apply(
    count_replacement_chars
)

metadata["raw_char_count"] = metadata["raw_text"].str.len()

metadata["raw_word_count"] = metadata["raw_text"].apply(
    lambda x: len(x.split())
)

print("Replacement character summary:")
display(metadata["replacement_char_count"].describe())

print(
    "Documents with replacement characters:",
    (metadata["replacement_char_count"] > 0).sum()
)

print("\nDocuments with the most replacement characters:")
display(
    metadata.nlargest(10, "replacement_char_count")[
        ["filename", "encoding", "replacement_char_count"]
    ]
)

In [ ]:
# Stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)

# Preserve negation
stopwords -= {"no", "nor", "not", "never", "neither"}

# Generic diplomatic/procedural terms
un_procedural_stopwords = {
    "mr",
    "mrs",
    "madam",
    "chairman",
    "chairperson",
    "delegation",
    "delegate",
    "delegates",
    "session",
    "plenary"
}
stopwords.update(un_procedural_stopwords)

ceremonial_candidates = {
    "congratulations",
    "congratulate",
    "pleasure",
    "appreciation",
    "behalf",
    "occasion",
    "thanks",
    "wish"
}
stopwords.update(ceremonial_candidates)

# Explicitly retain these because they may form meaningful phrases
stopwords -= {
    "assembly",
    "secretary",
    "president"
}

print(f"Total stopwords: {len(stopwords):,}")

## Clean and Tokenize the Speeches

In [ ]:
def tokenize_for_phrases(text):
    """
    Normalize and tokenize text while preserving stopwords.
    Used for bigram/trigram detection.
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Preserve dotted acronyms: U.S. -> u_s, U.N. -> u_n
    text = re.sub(
        r"\b(?:[a-z]\.){2,}",
        lambda m: m.group(0).rstrip(".").replace(".", "_"),
        text
    )

    return re.findall(
        r"[a-z]+(?:_[a-z]+)*(?:'[a-z]+)?",
        text
    )


# Preserve stopwords for phrase detection
metadata["phrase_tokens"] = metadata["raw_text"].apply(
    tokenize_for_phrases
)

# Original cleaned tokens for general analysis
metadata["tokens"] = metadata["phrase_tokens"].apply(
    lambda tokens: [
        token for token in tokens
        if token not in stopwords and len(token.replace("_", "")) > 1
    ]
)

metadata["clean_text"] = metadata["tokens"].apply(
    lambda tokens: " ".join(tokens)
)

metadata["clean_word_count"] = metadata["tokens"].apply(len)

display(
    metadata[
        ["filename", "raw_word_count", "clean_word_count"]
    ].head(10)
)

In [ ]:
from collections import Counter

bigram_counts = Counter()
bigram_documents = Counter()

for tokens in metadata["phrase_tokens"]:
    bigrams = list(zip(tokens[:-1], tokens[1:]))
    bigram_counts.update(bigrams)

    # Count each bigram once per speech
    bigram_documents.update(set(bigrams))


top_bigrams = pd.DataFrame(
    [
        {
            "bigram": " ".join(bigram),
            "frequency": count,
            "document_frequency": bigram_documents[bigram]
        }
        for bigram, count in bigram_counts.most_common(100)
        if not all(token in stopwords for token in bigram)
    ]
)

display(top_bigrams.head(50))

In [ ]:
# Compare before and after preprocessing
# Compare a randomly selected speech
sample = metadata.sample(1, random_state=42).iloc[0]

print("FILE:", sample["filename"])

print("\n--- ORIGINAL TEXT ---\n")
print(sample["raw_text"][:1200])

print("\n--- CLEANED TEXT ---\n")
print(sample["clean_text"][:1200])

print("\n--- FIRST 50 TOKENS ---\n")
print(sample["tokens"][:50])

In [ ]:
# Vocabulary after preprocessing
# Combine tokens across the entire corpus
all_tokens = [
    token
    for document_tokens in metadata["tokens"]
    for token in document_tokens
]

word_counts = Counter(all_tokens)

vocabulary = pd.DataFrame(
    word_counts.most_common(50),
    columns=["word", "frequency"]
)

print(f"Total tokens: {len(all_tokens):,}")
print(f"Unique vocabulary terms: {len(word_counts):,}")

display(vocabulary)

In [ ]:
# Preprocessing statistics
metadata["token_retention_pct"] = (
    metadata["clean_word_count"] /
    metadata["raw_word_count"].replace(0, np.nan)
) * 100

print("Preprocessing summary:")
display(
    metadata[
        [
            "raw_word_count",
            "clean_word_count",
            "token_retention_pct"
        ]
    ].describe()
)

print("\nDocuments with no remaining tokens:")
print((metadata["clean_word_count"] == 0).sum())

print("\nShortest cleaned speeches:")
display(
    metadata.nsmallest(10, "clean_word_count")[
        ["filename", "year", "country_code", "clean_word_count"]
    ]
)

## Save processed corpus

In [ ]:
output_path = PROCESSED_DIR / "ungdc_preprocessed.parquet"

columns_to_save = [
    "filename",
    "country_code",
    "session",
    "year",
    "file_path",
    "encoding",
    "raw_word_count",
    "clean_word_count",
    "replacement_char_count",
    "raw_text",
    "phrase_tokens",
    "clean_text",
    "tokens"
]

processed = metadata[columns_to_save].copy()

processed.to_parquet(
    output_path,
    index=False,
    engine="pyarrow"
)

print(f"Saved {len(processed):,} speeches to:")
print(output_path)

In [ ]:
# Validate
# Read the saved dataset back in
check = pd.read_parquet(output_path)

print(f"Saved documents: {len(check):,}")
print(f"Unique countries/entities: {check['country_code'].nunique():,}")
print(f"Year range: {check['year'].min()}–{check['year'].max()}")
print(f"Total cleaned tokens: {check['clean_word_count'].sum():,}")

display(check.head(3))